# T030 - drift stationarity screen

LightGBM on T026 wide features, holdout from 2026-10-27 18:00 (feature4 rollover, like test): keep dates / drop dates / stationary (item age, per-hour counter percentiles) / stationary + item profiles. CPU only, no submission.

In [ ]:
import json
import os
from pathlib import Path
import subprocess
import sys

sources = {'feature_screen.py': '"""Local LightGBM screen of request-context, train+test count and target-encoding features.\n\nTrains on rows before the original primary cutoff and scores the original\n307,528-row holdout. request_id is used only for grouping, never as an input.\n"""\nimport argparse\nfrom datetime import datetime\nimport gc\nimport json\nfrom pathlib import Path\nimport time\n\nimport lightgbm as lgb\nimport numpy as np\nimport polars as pl\nfrom sklearn.metrics import roc_auc_score\nfrom sklearn.model_selection import GroupKFold\n\nCUTOFF = datetime.fromisoformat("2026-10-28T08:06:52.397000")\nUSER, ITEM = "feature5", "feature18"\nLOW_CATEGORICAL = [f"feature{i}" for i in [2, 4, 6, 7, 8, 9, 10, 11, 12, 13, 14, 16, 17, 19, 21, 22, 25]]\nCOUNT_KEYS = [USER, ITEM, "feature23", "feature20", "feature19", "feature9", "feature12"]\nENCODED = [ITEM, "feature23", "feature20", "feature19", "feature9", "feature12", "feature21", USER]\nRANKED = ["feature418", "feature419", "feature411", "feature410", "feature430",\n          "feature431", "feature417", "feature352", "feature346", "feature480"]\nSMOOTHING = 20.0\n\n\ndef scalar_numeric(schema):\n    skip = {"feature1", "feature3"}\n    return [c for c, t in schema.items()\n            if c.startswith("feature") and c not in skip\n            and str(t) in {"Float32", "Int64", "Int8"}]\n\n\ndef time_seconds(column):\n    return pl.col(column).dt.epoch("ms").cast(pl.Float64) / 1000.0\n\n\ndef count_features(keys):\n    """Label-free counts over the complete train+test population."""\n    frame = keys.with_columns(time_seconds("feature1").alias("t"))\n    exprs = []\n    for key in COUNT_KEYS:\n        exprs += [pl.len().over(key).alias(f"cnt_{key}"),\n                  pl.col("request_id").n_unique().over(key).alias(f"req_{key}")]\n    frame = frame.with_columns(exprs)\n    pair = [USER, ITEM]\n    frame = frame.with_columns(\n        pl.len().over(pair).alias("cnt_user_item"),\n        pl.col("t").min().over(ITEM).alias("item_first"),\n        pl.col("t").max().over(ITEM).alias("item_last"),\n        pl.col("t").min().over(USER).alias("user_first"),\n        pl.col("t").max().over(USER).alias("user_last"),\n        pl.len().over([ITEM, pl.col("feature1").dt.truncate("1h")]).alias("cnt_item_hour"),\n        pl.len().over([ITEM, pl.col("feature1").dt.truncate("1d")]).alias("cnt_item_day"),\n    )\n    requests = (\n        frame.group_by("request_id").agg(pl.col(USER).first(), pl.col("t").first())\n        .sort([USER, "t", "request_id"])\n        .with_columns(\n            pl.int_range(pl.len()).over(USER).alias("user_req_index"),\n            (pl.col("t") - pl.col("t").shift(1).over(USER)).alias("user_prev_gap"),\n            (pl.col("t").shift(-1).over(USER) - pl.col("t")).alias("user_next_gap"),\n        )\n        .select("request_id", "user_req_index", "user_prev_gap", "user_next_gap")\n    )\n    frame = frame.join(requests, on="request_id", how="left")\n    pairs = (\n        frame.select(USER, ITEM, "t").unique().sort([USER, ITEM, "t"])\n        .with_columns(\n            pl.int_range(pl.len()).over([USER, ITEM]).alias("user_item_index"),\n            (pl.col("t") - pl.col("t").shift(1).over([USER, ITEM])).alias("user_item_prev_gap"),\n            (pl.col("t").shift(-1).over([USER, ITEM]) - pl.col("t")).alias("user_item_next_gap"),\n        )\n    )\n    frame = frame.join(pairs, on=[USER, ITEM, "t"], how="left")\n    frame = frame.with_columns(\n        (pl.col("req_feature5") - pl.col("user_req_index") - 1).alias("user_reqs_after"),\n        (pl.col("t") - pl.col("item_first")).alias("item_age"),\n        (pl.col("item_last") - pl.col("t")).alias("item_remaining"),\n        (pl.col("t") - pl.col("user_first")).alias("user_since_first"),\n        (pl.col("user_last") - pl.col("t")).alias("user_until_last"),\n    ).drop("item_first", "item_last", "user_first", "user_last")\n    names = [c for c in frame.columns if c not in keys.columns and c != "t"]\n    return frame.select("row", *names), names\n\n\ndef request_features(frame):\n    """Within-request context; frame is one split population with numeric columns."""\n    group = "request_id"\n    exprs = [pl.len().over(group).alias("rq_size"),\n             pl.col(ITEM).n_unique().over(group).alias("rq_items"),\n             pl.len().over([group, ITEM]).alias("rq_item_dup")]\n    for c in RANKED:\n        exprs += [\n            (pl.col(c).rank("average").over(group) / pl.len().over(group)).alias(f"rq_rank_{c}"),\n            (pl.col(c) - pl.col(c).mean().over(group)).alias(f"rq_dev_{c}"),\n            (pl.col(c) - pl.col(c).max().over(group)).alias(f"rq_gap_{c}"),\n        ]\n    out = frame.select("row", *exprs)\n    return out, [c for c in out.columns if c != "row"]\n\n\ndef target_encode(train, valid, column, folds):\n    prior = float(train["watched"].mean())\n    y = train["watched"].to_numpy().astype(np.float64)\n    keys = train[column]\n    oof = np.empty(len(train))\n    support = np.empty(len(train))\n    for fit, apply in folds:\n        stats = (pl.DataFrame({"k": keys[fit], "y": y[fit]})\n                 .group_by("k").agg(pl.col("y").sum().alias("s"), pl.len().alias("n")))\n        joined = pl.DataFrame({"k": keys[apply]}).join(stats, on="k", how="left").fill_null(0)\n        s, n = joined["s"].to_numpy(), joined["n"].to_numpy()\n        oof[apply] = (s + SMOOTHING * prior) / (n + SMOOTHING)\n        support[apply] = n\n    stats = (pl.DataFrame({"k": keys, "y": y})\n             .group_by("k").agg(pl.col("y").sum().alias("s"), pl.len().alias("n")))\n    joined = pl.DataFrame({"k": valid[column]}).join(stats, on="k", how="left").fill_null(0)\n    s, n = joined["s"].to_numpy(), joined["n"].to_numpy()\n    held = (s + SMOOTHING * prior) / (n + SMOOTHING)\n    return (oof, support), (held, n)\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--data", type=Path, default=Path("data"))\n    parser.add_argument("--output", type=Path, default=Path("artifacts/local-t023"))\n    parser.add_argument("--variants", default="base,request,counts,target,all")\n    parser.add_argument("--learning-rate", type=float, default=0.05)\n    parser.add_argument("--rounds", type=int, default=4000)\n    parser.add_argument("--threads", type=int, default=8)\n    parser.add_argument("--rows", type=int, default=0, help="Keep 1/N of requests for smoke tests")\n    args = parser.parse_args()\n    args.output.mkdir(parents=True, exist_ok=True)\n    start = time.time()\n    schema = pl.scan_parquet(args.data / "train.parquet").collect_schema()\n    numeric = scalar_numeric(schema)\n    key_columns = ["request_id", "feature1", *sorted(set(COUNT_KEYS + ENCODED + LOW_CATEGORICAL), key=lambda c: int(c[7:]))]\n    train = pl.scan_parquet(args.data / "train.parquet").with_row_index("row")\n    if args.rows:\n        train = train.filter(pl.col("request_id").hash(7) % args.rows == 0)\n    train = train.select("row", "watched", "feature3", *key_columns, *numeric).collect()\n    test_keys = pl.scan_parquet(args.data / "test.parquet").select(*key_columns)\n    if args.rows:\n        test_keys = test_keys.filter(pl.col("request_id").hash(7) % args.rows == 0)\n    test_keys = test_keys.collect()\n    offset = int(train["row"].max()) + 1\n    keys = pl.concat([\n        train.select("row", *key_columns),\n        test_keys.with_row_index("row", offset=offset).with_columns(pl.col("row").cast(train["row"].dtype)),\n    ])\n    del test_keys\n    counts, count_names = count_features(keys)\n    del keys\n    gc.collect()\n    train = train.join(counts, on="row", how="left")\n    del counts\n    rq, rq_names = request_features(train.select("row", "request_id", ITEM, *RANKED))\n    train = train.join(rq, on="row", how="left")\n    del rq\n    is_valid = (train["feature1"] >= CUTOFF).to_numpy()\n    fit_rows, valid_rows = train.filter(~pl.Series(is_valid)), train.filter(pl.Series(is_valid))\n    del train\n    gc.collect()\n    folds = list(GroupKFold(5).split(np.zeros(len(fit_rows)), groups=fit_rows["request_id"].to_numpy()))\n    te_fit, te_valid, te_names = {}, {}, []\n    for column in ENCODED:\n        (a, an), (b, bn) = target_encode(fit_rows, valid_rows, column, folds)\n        te_fit[f"te_{column}"], te_valid[f"te_{column}"] = a, b\n        te_fit[f"ten_{column}"], te_valid[f"ten_{column}"] = an, bn\n        te_names += [f"te_{column}", f"ten_{column}"]\n    categories = {}\n    for c in LOW_CATEGORICAL:\n        vocabulary = fit_rows[c].drop_nulls().unique().sort()\n        categories[c] = vocabulary\n    base = ["feature3", *numeric, *LOW_CATEGORICAL]\n    groups = {"base": base, "request": base + rq_names, "counts": base + count_names,\n              "target": base + te_names, "all": base + rq_names + count_names + te_names}\n\n    def matrix(frame, te, columns):\n        parts = []\n        for c in columns:\n            if c in te:\n                parts.append(te[c].astype(np.float32))\n            elif c in categories:\n                codes = frame.select(pl.col(c).cast(pl.Enum(categories[c].to_list()), strict=False)\n                                     .to_physical().cast(pl.Float32))[c].to_numpy()\n                parts.append(codes)\n            else:\n                parts.append(frame[c].cast(pl.Float32).to_numpy())\n        return np.ascontiguousarray(np.column_stack(parts), dtype=np.float32)\n\n    y_fit = fit_rows["watched"].to_numpy()\n    y_valid = valid_rows["watched"].to_numpy()\n    valid_ids = valid_rows["row"].to_numpy()\n    results = {}\n    for name in args.variants.split(","):\n        columns = groups[name]\n        began = time.time()\n        x_fit = matrix(fit_rows, te_fit, columns)\n        x_valid = matrix(valid_rows, te_valid, columns)\n        categorical = [i for i, c in enumerate(columns) if c in categories]\n        train_set = lgb.Dataset(x_fit, y_fit, feature_name=columns,\n                                categorical_feature=categorical, free_raw_data=True)\n        valid_set = lgb.Dataset(x_valid, y_valid, reference=train_set)\n        params = {\n            "objective": "binary", "metric": "auc", "learning_rate": args.learning_rate,\n            "num_leaves": 127, "min_data_in_leaf": 200, "feature_fraction": 0.5,\n            "bagging_fraction": 0.8, "bagging_freq": 1, "lambda_l2": 10,\n            "max_bin": 127, "seed": 42, "num_threads": args.threads, "verbosity": -1,\n            "cat_smooth": 20, "max_cat_to_onehot": 8,\n        }\n        log = {}\n        model = lgb.train(params, train_set, args.rounds, valid_sets=[valid_set], valid_names=["holdout"],\n                          callbacks=[lgb.early_stopping(200, verbose=False), lgb.record_evaluation(log),\n                                     lgb.log_evaluation(250)])\n        prediction = model.predict(x_valid, num_iteration=model.best_iteration)\n        auc = float(roc_auc_score(y_valid, prediction))\n        curve = log["holdout"]["auc"]\n        importance = sorted(zip(columns, model.feature_importance("gain")), key=lambda p: -p[1])[:30]\n        results[name] = {\n            "holdout_auc": auc, "best_iteration": model.best_iteration,\n            "auc_at_140": curve[min(139, len(curve) - 1)], "features": len(columns),\n            "seconds": time.time() - began,\n            "top_gain": [[c, float(g)] for c, g in importance],\n        }\n        np.save(args.output / f"valid-{name}.npy", prediction.astype(np.float32))\n        print(name, json.dumps({k: v for k, v in results[name].items() if k != "top_gain"}), flush=True)\n        print(" top:", [c for c, _ in importance[:15]], flush=True)\n        (args.output / "results.json").write_text(json.dumps({\n            "cutoff": CUTOFF.isoformat(), "valid_rows": int(len(y_valid)),\n            "fit_rows": int(len(y_fit)), "early_stopping_on_holdout": True,\n            "results": results, "seconds": time.time() - start}, indent=2) + "\\n")\n        del x_fit, x_valid, train_set, valid_set, model\n        gc.collect()\n    np.save(args.output / "valid-row-ids.npy", valid_ids)\n\n\nif __name__ == "__main__":\n    main()\n', 'request_screen.py': '"""T026 local LightGBM screen: wider within-request context features.\n\nSame protocol as T023 (train before the original cutoff, score the original\n307,528-row holdout, early stopping on the holdout for every variant, so deltas\nare comparable). Wide features are chosen by gain from a fixed-budget base\nmodel fitted on training rows only; the holdout is never used to choose them.\n"""\nimport argparse\nimport gc\nimport json\nfrom pathlib import Path\nimport time\n\nimport lightgbm as lgb\nimport numpy as np\nimport polars as pl\nfrom sklearn.metrics import roc_auc_score\n\nfrom feature_screen import CUTOFF, ITEM, LOW_CATEGORICAL, RANKED, scalar_numeric\n\nPARAMS = {\n    "objective": "binary", "metric": "auc", "num_leaves": 127, "min_data_in_leaf": 200,\n    "feature_fraction": 0.5, "bagging_fraction": 0.8, "bagging_freq": 1, "lambda_l2": 10,\n    "max_bin": 127, "seed": 42, "verbosity": -1, "cat_smooth": 20, "max_cat_to_onehot": 8,\n}\n\n\ndef context(columns, stats=()):\n    group = "request_id"\n    size = pl.len().over(group)\n    exprs = [size.alias("rq_size"), pl.col(ITEM).n_unique().over(group).alias("rq_items"),\n             pl.len().over([group, ITEM]).alias("rq_item_dup")]\n    for c in columns:\n        exprs += [(pl.col(c).rank("average").over(group) / size).alias(f"rq_rank_{c}"),\n                  (pl.col(c) - pl.col(c).mean().over(group)).alias(f"rq_dev_{c}"),\n                  (pl.col(c) - pl.col(c).max().over(group)).alias(f"rq_gap_{c}")]\n    for c in stats:\n        exprs += [pl.col(c).mean().over(group).alias(f"rq_mean_{c}"),\n                  pl.col(c).std().over(group).alias(f"rq_std_{c}")]\n    return [e.cast(pl.Float32) for e in exprs]\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--data", type=Path, default=Path("data"))\n    parser.add_argument("--output", type=Path, default=Path("artifacts/local-t026"))\n    parser.add_argument("--variants", default="control,no_size,wide,wide_stats")\n    parser.add_argument("--wide", type=int, default=50)\n    parser.add_argument("--threads", type=int, default=6)\n    parser.add_argument("--rows", type=int, default=0, help="Keep 1/N of requests for smoke tests")\n    args = parser.parse_args()\n    args.output.mkdir(parents=True, exist_ok=True)\n    start = time.time()\n    numeric = scalar_numeric(pl.scan_parquet(args.data / "train.parquet").collect_schema())\n    frame = pl.scan_parquet(args.data / "train.parquet")\n    if args.rows:\n        frame = frame.filter(pl.col("request_id").hash(7) % args.rows == 0)\n    frame = frame.select("watched", "feature1", "feature3", "request_id", ITEM,\n                         *dict.fromkeys([*LOW_CATEGORICAL, *numeric])).collect()\n    is_valid = (frame["feature1"] >= CUTOFF).to_numpy()\n    categories = {c: frame.filter(pl.Series(~is_valid))[c].drop_nulls().unique().sort().to_list()\n                  for c in LOW_CATEGORICAL}\n    base = ["feature3", *numeric, *LOW_CATEGORICAL]\n\n    def matrix(source, columns):\n        parts = []\n        for c in columns:\n            column = pl.col(c)\n            if c in categories:\n                column = column.cast(pl.Enum(categories[c]), strict=False).to_physical()\n            parts.append(source.select(column.cast(pl.Float32))[c].to_numpy())\n        return np.ascontiguousarray(np.column_stack(parts), dtype=np.float32)\n\n    y = frame["watched"].to_numpy()\n    y_fit, y_valid = y[~is_valid], y[is_valid]\n    categorical = lambda columns: [i for i, c in enumerate(columns) if c in categories]\n    x = matrix(frame.filter(pl.Series(~is_valid)), base)\n    selector = lgb.train({**PARAMS, "learning_rate": .05, "num_threads": args.threads},\n                         lgb.Dataset(x, y_fit, feature_name=base, categorical_feature=categorical(base)), 150)\n    gains = dict(zip(base, selector.feature_importance("gain")))\n    del x, selector\n    gc.collect()\n    candidates = [c for c in numeric if c not in categories and c not in {"feature3"}]\n    wide = sorted(candidates, key=lambda c: -gains[c])[:args.wide]\n    wide = list(dict.fromkeys([*RANKED, *wide]))\n    print(f"wide context over {len(wide)} features; top: {wide[10:20]}", flush=True)\n    frame = frame.with_columns(context(wide, wide[:10]))\n    control = ["rq_size", "rq_items", "rq_item_dup"] + [f"rq_{k}_{c}" for c in RANKED for k in ["rank", "dev", "gap"]]\n    wide_names = ["rq_size", "rq_items", "rq_item_dup"] + [f"rq_{k}_{c}" for c in wide for k in ["rank", "dev", "gap"]]\n    groups = {\n        "control": base + control,\n        "no_size": base + [c for c in control if c != "rq_size"],\n        "wide": base + wide_names,\n        "wide_stats": base + wide_names + [f"rq_{k}_{c}" for c in wide[:10] for k in ["mean", "std"]],\n    }\n    fit_rows, valid_rows = frame.filter(pl.Series(~is_valid)), frame.filter(pl.Series(is_valid))\n    del frame\n    gc.collect()\n    results = {}\n    for name in args.variants.split(","):\n        columns, began = groups[name], time.time()\n        train_set = lgb.Dataset(matrix(fit_rows, columns), y_fit, feature_name=columns,\n                                categorical_feature=categorical(columns), free_raw_data=True)\n        x_valid = matrix(valid_rows, columns)\n        valid_set = lgb.Dataset(x_valid, y_valid, reference=train_set)\n        model = lgb.train({**PARAMS, "learning_rate": .05, "num_threads": args.threads}, train_set, 4000,\n                          valid_sets=[valid_set], valid_names=["holdout"],\n                          callbacks=[lgb.early_stopping(200, verbose=False), lgb.log_evaluation(250)])\n        auc = float(roc_auc_score(y_valid, model.predict(x_valid, num_iteration=model.best_iteration)))\n        top = sorted(zip(columns, model.feature_importance("gain")), key=lambda p: -p[1])[:30]\n        results[name] = {"holdout_auc": auc, "best_iteration": model.best_iteration,\n                         "features": len(columns), "seconds": time.time() - began,\n                         "top_gain": [[c, float(g)] for c, g in top]}\n        print(name, json.dumps({k: v for k, v in results[name].items() if k != "top_gain"}), flush=True)\n        (args.output / "results.json").write_text(json.dumps({\n            "cutoff": CUTOFF.isoformat(), "valid_rows": int(len(y_valid)), "fit_rows": int(len(y_fit)),\n            "early_stopping_on_holdout": True, "wide_features": wide, "results": results,\n            "seconds": time.time() - start}, indent=2) + "\\n")\n        del train_set, valid_set, x_valid, model\n        gc.collect()\n\n\nif __name__ == "__main__":\n    main()\n', 'drift_context_screen.py': '"""T030 local screens: train/test drift and stationary replacements.\n\nadversarial / adversarial_no_dates: LightGBM separating train rows from test rows on\nthe model inputs (no time column); reports AUC and the most drifting features.\n\nstationarity: on the T026 wide request features and a holdout starting where\nfeature4 (shifted date) rolls over, like test, compares\n  keep_dates           current inputs\n  drop_dates           without feature2/feature4 (calendar dates never seen in test)\n  stationary           drop_dates, item age instead of upload timestamps feature58/59,\n                       per-hour percentiles instead of counters whose level swings\n                       between time windows (counters.json, chosen on train+test medians)\n  stationary_profiles  plus label-free item/key profiles over train+test\nLabels are never used outside the training rows.\n"""\nimport argparse\nimport gc\nimport json\nfrom pathlib import Path\nimport time\n\nimport lightgbm as lgb\nimport numpy as np\nimport polars as pl\nfrom sklearn.metrics import roc_auc_score\n\nfrom feature_screen import CUTOFF, ITEM, LOW_CATEGORICAL, scalar_numeric\nfrom request_screen import PARAMS, context\n\nKEYS = [ITEM, "feature23", "feature20", "feature19"]\nDATES = ["feature2", "feature4"]\n\n\ndef adversarial(data, output, threads, rows=400_000, exclude=()):\n    numeric = scalar_numeric(pl.scan_parquet(data / "train.parquet").collect_schema())\n    columns = [c for c in ["feature3", *numeric, *LOW_CATEGORICAL] if c not in set(exclude)]\n    frames = []\n    for label, name in enumerate(["train", "test"]):\n        frame = pl.scan_parquet(data / f"{name}.parquet").select(columns).collect()\n        frames.append(frame.sample(min(rows, frame.height), seed=42).with_columns(pl.lit(label).alias("_is_test")))\n    frame = pl.concat(frames)\n    del frames\n    categories = {c: frame[c].drop_nulls().unique().sort().to_list() for c in LOW_CATEGORICAL if c in columns}\n    x = np.column_stack([\n        frame.select((pl.col(c).cast(pl.Enum(categories[c])).to_physical() if c in categories\n                      else pl.col(c)).cast(pl.Float32))[c].to_numpy() for c in columns\n    ]).astype(np.float32)\n    y = frame["_is_test"].to_numpy()\n    order = np.random.default_rng(0).permutation(len(y))\n    fit, valid = order[: int(0.8 * len(y))], order[int(0.8 * len(y)):]\n    categorical = [i for i, c in enumerate(columns) if c in categories]\n    model = lgb.train({**PARAMS, "learning_rate": 0.1, "num_threads": threads},\n                      lgb.Dataset(x[fit], y[fit], feature_name=columns, categorical_feature=categorical),\n                      300)\n    auc = float(roc_auc_score(y[valid], model.predict(x[valid])))\n    gains = sorted(zip(columns, model.feature_importance("gain")), key=lambda p: -p[1])\n    total = sum(g for _, g in gains)\n    top = [[c, round(float(g / total), 4)] for c, g in gains[:30]]\n    single = {}\n    for name, _ in top[:30]:\n        index = columns.index(name)\n        values = np.nan_to_num(x[valid, index], nan=-1e9)\n        single[name] = float(max(roc_auc_score(y[valid], values), 1 - roc_auc_score(y[valid], values)))\n    report = {"auc": auc, "rows_per_side": rows, "excluded": list(exclude), "top_gain_share": top, "single_feature_auc": single}\n    (output / f"adversarial{\'-no-\' + \'-\'.join(exclude) if exclude else \'\'}.json").write_text(json.dumps(report, indent=2) + "\\n")\n    print("adversarial AUC", round(auc, 4), flush=True)\n    for name, share in top[:25]:\n        print(f"  {name:12s} gain {share:.3f}  single AUC {single.get(name, 0):.3f}", flush=True)\n\n\ndef transductive_features(data, profiled, counters):\n    """Label-free features computed over train+test together, returned for train rows.\n\n    Item profiles: each key\'s mean of profiled features and the row\'s deviation.\n    Stationary: item age instead of upload timestamps (feature58 s, feature59 ms) and\n    per-hour percentiles of counters whose level swings between time windows.\n    """\n    needed = list(dict.fromkeys(["feature1", "feature58", "feature59", *KEYS, *profiled, *counters]))\n    both = pl.concat([\n        pl.scan_parquet(data / f"{name}.parquet").select(needed).with_row_index("_row")\n        .with_columns(pl.lit(name).alias("_split")).collect()\n        for name in ["train", "test"]\n    ], how="vertical_relaxed")\n    hour = pl.col("feature1").dt.truncate("1h")\n    now = pl.col("feature1").dt.epoch("ms").cast(pl.Float64) / 1000\n    stationary = [((now - pl.col("feature58")) / 3600).alias("st_age58_h"),\n                  ((now - pl.col("feature59") / 1000) / 3600).alias("st_age59_h")]\n    stationary += [(pl.col(c).rank("average").over(hour) / pl.col(c).count().over(hour)).alias(f"st_hp_{c}")\n                   for c in counters]\n    profile = []\n    for key in KEYS:\n        profile.append((pl.len().over([key, hour]) / pl.len().over(hour)).alias(f"pf_share_{key}"))\n        for column in profiled:\n            mean = pl.col(column).mean().over(key)\n            profile += [mean.alias(f"pf_mean_{key}_{column}"), (pl.col(column) - mean).alias(f"pf_dev_{key}_{column}")]\n    exprs = [e.cast(pl.Float32) for e in stationary + profile]\n    both = both.with_columns(exprs)\n    names = lambda group: [e.meta.output_name() for e in group]\n    train = both.filter(pl.col("_split") == "train").sort("_row").select(names(stationary + profile))\n    return train, names(stationary), names(profile)\n\n\ndef wide_frame(data, wide, extra=None):\n    numeric = scalar_numeric(pl.scan_parquet(data / "train.parquet").collect_schema())\n    frame = pl.scan_parquet(data / "train.parquet").select(\n        "watched", "feature1", "feature3", "request_id", ITEM,\n        *dict.fromkeys([*LOW_CATEGORICAL, *numeric])).collect()\n    if extra is not None:\n        if extra.height != frame.height:\n            raise ValueError("Transductive rows do not align with train rows")\n        frame = pl.concat([frame, extra], how="horizontal")\n    wide_names = ["rq_size", "rq_items", "rq_item_dup"] + [f"rq_{k}_{c}" for c in wide for k in ["rank", "dev", "gap"]]\n    return frame.with_columns(context(wide)), ["feature3", *numeric, *LOW_CATEGORICAL, *wide_names]\n\n\ndef fit_variants(holder, cutoff, variants, threads, output, name):\n    """holder: one-item list owning the frame, so it is freed once matrices are built."""\n    started = time.time()\n    frame = holder.pop()\n    is_valid = (frame["feature1"] >= cutoff).to_numpy()\n    categories = {c: frame.filter(pl.Series(~is_valid))[c].drop_nulls().unique().sort().to_list()\n                  for c in LOW_CATEGORICAL}\n    union = list(dict.fromkeys(c for columns in variants.values() for c in columns))\n    y = frame["watched"].to_numpy()\n    x_fit = np.empty((int((~is_valid).sum()), len(union)), dtype=np.float32)\n    x_valid = np.empty((int(is_valid.sum()), len(union)), dtype=np.float32)\n    for i, c in enumerate(union):\n        column = pl.col(c)\n        if c in categories:\n            column = column.cast(pl.Enum(categories[c]), strict=False).to_physical()\n        values = frame.select(column.cast(pl.Float32))[c].to_numpy()\n        x_fit[:, i], x_valid[:, i] = values[~is_valid], values[is_valid]\n    del frame\n    gc.collect()\n    print(f"{name}: cutoff {cutoff}, fit {len(x_fit):,}, holdout {len(x_valid):,}, columns {len(union)}", flush=True)\n    path = output / f"{name}.json"\n    results = {}\n    for label, columns in variants.items():\n        began = time.time()\n        index = [union.index(c) for c in columns]\n        categorical = [i for i, c in enumerate(columns) if c in categories]\n        train_set = lgb.Dataset(x_fit[:, index], y[~is_valid], feature_name=columns,\n                                categorical_feature=categorical, free_raw_data=True)\n        valid_x = x_valid[:, index]\n        valid_set = lgb.Dataset(valid_x, y[is_valid], reference=train_set)\n        model = lgb.train({**PARAMS, "learning_rate": .05, "num_threads": threads}, train_set, 4000,\n                          valid_sets=[valid_set], valid_names=["holdout"],\n                          callbacks=[lgb.early_stopping(200, verbose=False), lgb.log_evaluation(250)])\n        auc = float(roc_auc_score(y[is_valid], model.predict(valid_x, num_iteration=model.best_iteration)))\n        gains = sorted(zip(columns, model.feature_importance("gain")), key=lambda p: -p[1])\n        total = sum(g for _, g in gains)\n        results[label] = {"holdout_auc": auc, "best_iteration": model.best_iteration,\n                          "features": len(columns), "seconds": time.time() - began,\n                          "top_gain": [[c, float(g / total)] for c, g in gains[:40]]}\n        print(label, json.dumps({k: v for k, v in results[label].items() if k != "top_gain"}), flush=True)\n        path.write_text(json.dumps({"cutoff": str(cutoff), "holdout_rows": int(len(valid_x)),\n                                    "variants": {k: v for k, v in variants.items()},\n                                    "results": results, "seconds": time.time() - started}, indent=2) + "\\n")\n        del train_set, valid_set, valid_x, model\n        gc.collect()\n    return results\n\n\ndef stationarity(data, output, threads, wide_from, counters_from, profile_count, cutoff):\n    wide = json.loads(wide_from.read_text())["wide_features"]\n    counters = json.loads(counters_from.read_text())\n    extra, stationary, profile = transductive_features(data, wide[:profile_count], counters)\n    holder = [None]\n    holder[0], columns = wide_frame(data, wide, extra)\n    del extra\n    replaced = set(DATES) | {"feature58", "feature59"} | set(counters)\n    stable = [c for c in columns if c not in replaced] + stationary\n    fit_variants(holder, cutoff, {\n        "keep_dates": columns,\n        "drop_dates": [c for c in columns if c not in DATES],\n        "stationary": stable,\n        "stationary_profiles": stable + profile,\n    }, threads, output, f"stationarity-{cutoff:%m%dT%H%M}")\n\n\ndef main():\n    from datetime import datetime\n\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--data", type=Path, default=Path("data"))\n    parser.add_argument("--output", type=Path, default=Path("artifacts/local-t030"))\n    parser.add_argument("--stages", default="adversarial,adversarial_no_dates,stationarity")\n    parser.add_argument("--wide-from", type=Path, default=Path("artifacts/local-t026/results.json"))\n    parser.add_argument("--counters-from", type=Path, default=Path("artifacts/local-t030/counters.json"))\n    parser.add_argument("--profile-count", type=int, default=12)\n    parser.add_argument("--cutoff", default="2026-10-27T18:00:00",\n                        help="Holdout start; 18:00 is where feature4 (shifted date) rolls over, like test")\n    parser.add_argument("--threads", type=int, default=6)\n    args = parser.parse_args()\n    args.output.mkdir(parents=True, exist_ok=True)\n    cutoff = datetime.fromisoformat(args.cutoff)\n    for stage in args.stages.split(","):\n        if stage == "adversarial":\n            adversarial(args.data, args.output, args.threads)\n        elif stage == "adversarial_no_dates":\n            adversarial(args.data, args.output, args.threads, exclude=DATES)\n        elif stage == "stationarity":\n            stationarity(args.data, args.output, args.threads, args.wide_from, args.counters_from,\n                         args.profile_count, cutoff)\n        else:\n            raise ValueError(f"Unknown stage: {stage}")\n        gc.collect()\n\n\nif __name__ == "__main__":\n    main()\n'}
frozen = Path("/kaggle/working/frozen-source")
frozen.mkdir()
for name, source in sources.items():
    (frozen / name).write_text(source)
(frozen / "t026-wide.json").write_text('{"wide_features": ["feature418", "feature419", "feature411", "feature410", "feature430", "feature431", "feature417", "feature352", "feature346", "feature480", "feature54", "feature493", "feature358", "feature515", "feature490", "feature429", "feature492", "feature510", "feature334", "feature511", "feature103", "feature39", "feature416", "feature484", "feature503", "feature150", "feature356", "feature151", "feature53", "feature337", "feature512", "feature478", "feature428", "feature514", "feature37", "feature513", "feature357", "feature376", "feature502", "feature439", "feature479", "feature507", "feature487", "feature333", "feature50", "feature320", "feature117", "feature126", "feature491", "feature56", "feature153"]}')
(frozen / "counters.json").write_text('["feature103", "feature253", "feature254", "feature255", "feature256", "feature257", "feature258", "feature259", "feature260", "feature261", "feature262", "feature263", "feature264", "feature267", "feature268", "feature269", "feature270", "feature271", "feature272", "feature273", "feature274", "feature275", "feature276", "feature277", "feature278", "feature279", "feature280", "feature281", "feature282", "feature283", "feature284", "feature285", "feature286", "feature527", "feature570", "feature571"]')
data = [p.parent for p in Path("/kaggle/input").rglob("train.parquet") if (p.parent / "sample_submission.csv").is_file()]
if len(data) != 1:
    raise RuntimeError(f"Expected competition data: {data}")
import lightgbm, sklearn
print("versions", lightgbm.__version__, sklearn.__version__, os.cpu_count(), flush=True)
subprocess.run("free -m", shell=True, check=False)
monitor = subprocess.Popen("while true; do echo MEM $(date +%T) $(free -m | sed -n 2p); sleep 60; done", shell=True)
try:
    subprocess.run([
        sys.executable, str(frozen / "drift_context_screen.py"), "--data", str(data[0]), "--stages", "stationarity", "--counters-from", str(frozen / "counters.json"),
        "--wide-from", str(frozen / "t026-wide.json"), "--output", "/kaggle/working/t030",
        "--threads", str(os.cpu_count()),
    ], check=True, cwd=frozen)
finally:
    monitor.kill()
[print(p.name, p.read_text(), flush=True) for p in Path("/kaggle/working/t030").glob("*.json")]
